### Import Libraries

In [1]:
import re
import json
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import os
from dotenv import load_dotenv
load_dotenv()

import requests

In [2]:
# VERSION = "28_11_24"
VERSION = "27_01_25"

In [3]:
def get_dev_unique_values():
    url = "https://apim-gst-dev.azure-api.net/func-uiproperties-d-ussc-01/func-ui-properties"
    
    headers = {
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": "c2404e96bf0b471daf7f2b1091094fa1"
              }
    try:
        res = requests.post(url, headers=headers)
        res = res.json()
    except Exception as e:
        print(e)
        print(payload)
        
    return res
unique_values = get_dev_unique_values()

with open("unique_values_" + VERSION + ".json", "w") as fp:
    json.dump(unique_values['results'], fp)

In [4]:
unique_values['results']['UL']

[{'UL_PROPERTY': 'Minimum Thickness (mm)',
  'Min_Value': 0.1,
  'Max_Value': 6,
  'Avg_value': 3.05,
  'UNIT_OF_MEAS_SI': 'mm',
  'UL_PROPERTY_ID': 'UL_1',
  'UL_PROPERTY_UI': 'Minimum Thickness (mm)',
  'TEST_METHOD_NAME': None,
  'Categorical_Values': None,
  'SUB_PROPERTIES': [{'UL_SUB_PROPERTY_ID': '1200',
    'UL_SUB_PROPERTY': 'Relative Thermal Index - Mechanical Strength (RTI Str) (°C)',
    'UL_SUB_PROPERTY_UI': 'Relative Thermal Index - Mechanical Strength (RTI Str) (°C) UL 746B',
    'TEST_METHOD_NAME': 'UL 746B',
    'Min_Value': 50,
    'Max_Value': 240,
    'Avg_value': 145,
    'UNIT_OF_MEAS_SI': '°C'},
   {'UL_SUB_PROPERTY_ID': '1220',
    'UL_SUB_PROPERTY': 'Relative Thermal Index - Mechanical Impact (RTI Imp) (°C)',
    'UL_SUB_PROPERTY_UI': 'Relative Thermal Index - Mechanical Impact (RTI Imp) (°C) UL 746B',
    'TEST_METHOD_NAME': 'UL 746B',
    'Min_Value': 50,
    'Max_Value': 220,
    'Avg_value': 135,
    'UNIT_OF_MEAS_SI': '°C'},
   {'UL_SUB_PROPERTY_ID': '1240

In [5]:
unique_values['results'].keys()

dict_keys(['Property', 'Feature', 'UL', 'Certification', 'Auto_Approval', 'Brand', 'Polymer', 'Filler', 'Market', 'Industry_Group'])

In [6]:
connection_string = eval(os.getenv('connection_string'))
snowflake_connection_string = connection_string['ml-gst-dev-usscc-01']

parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0]
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    # database = 'ANALYTICS_QA', #database,
    database = 'ANALYTICS_DEV', #database,
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()



def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df

### Grades

In [7]:
grades = read_data_from_snowflake_table(cur,"""select distinct PRODUCT_CD from GST_Curated.SPT""")

all_grades = grades['product_cd'].unique().tolist()
all_grades = [value.lower() for value in all_grades]

    
len(all_grades)

1694

In [8]:
all_brands = [i.strip('®').lower() for i in unique_values["results"]["Brand"]]
len(all_brands)

19

In [9]:
extra_grades = ["dym", "eco-b", "eco b","eco-r", "eco r","esd", "fit", "frhr", "hfs", "hhr",
"hrlm", "hrt", "hsl", "hslr", "hte", "htn", "htr", "ice", 
"icf", "lcpa", "lds", "lof", "lof2", "med", "pcxxx", "pls/xt", 
"scxxx", "sea", "slidex", "wrf", "xap", "xap2", "xfr", "xgc"]

extra_grades = [i for i in extra_grades if i not in ['esd', 'lcpa', 'lds', 'med']]
all_grades.extend(extra_grades)

In [10]:
col = 'brand'
ignore_syn=[]
ignore_key=[]
positives = []
# SYNONYM data from dev database i.e. "analytics_dev". Refer to "DEFINED_NAME" and "SYNONYMS".
synonym_df = read_data_from_snowflake_table(cur,"""select * from SYNONYM""")
synonym_df.columns = [x.upper() if x.islower() else x for x in synonym_df.columns]
synonym_df = synonym_df.apply(lambda x: x.str.lower())

synonym_df = synonym_df[synonym_df['TYPE'] == col]

synonym_df2 = synonym_df[["DEFINED_NAME", "SYNONYMS"]].copy()
synonym_df2 = synonym_df2.dropna().reset_index(drop=True)
# synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : False if ";" in x else True)].reset_index(drop=True)
synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : True if ";" in x else False)].reset_index(drop=True)
brand_synonyms = synonym_df2.to_dict(orient='records')
brands_with_synonym = [item['DEFINED_NAME'] for item in brand_synonyms]
brand_synonyms  = {item['DEFINED_NAME']: item['SYNONYMS'] for item in brand_synonyms}

for k in brand_synonyms:
    if ";" in brand_synonyms[k]:
        brand_synonyms[k] = [x.strip() for x in  brand_synonyms[k].split(';') if x!=k]

# grades with brand synonym
grade_names_with_brand_synonym = []
for i in brands_with_synonym:
   brand_pattern = fr'^{re.escape(i)}\b'
   for grade in all_grades:
      if re.match(brand_pattern, grade):
         for s in brand_synonyms[i]:
            grade_names_with_brand_synonym.append(re.sub(brand_pattern, s, grade))

all_grades.extend(grade_names_with_brand_synonym)
all_grades = list(set(all_grades))
print(len(all_grades))

with open("all_grades_" + VERSION + ".json", "w") as fp:
    json.dump(all_grades, fp)

4228


In [11]:
synonym_df

,TYPE,DEFINED_NAME,SYNONYMS
2,brand,abistir,abistir
3,brand,amcel,amcel; am
4,brand,at,at
5,brand,ateva,ateva
6,brand,bexloy,bexloy
...,...,...,...
62,brand,vandar,vandar; va
63,brand,vectra,vectra; ve
64,brand,vitaldose,vitaldose
65,brand,zenite,zenite; ze


### Brands

In [12]:
brand_synonyms

{'amcel': ['am'],
 'blendfor': ['bf'],
 'blueridge': ['br', 'blue ridge'],
 'celanex': ['cx'],
 'celanyl': ['cnl'],
 'celapex': ['cl'],
 'celcon': ['cn'],
 'celstran': ['cs',
  'lft',
  'lfrt',
  'cs lft',
  'celstran lft',
  'pultrusion',
  'lftr'],
 'coolpoly': ['cp'],
 'crastin': ['cra'],
 'elvamide': ['mid'],
 'factor': ['fa'],
 'forflex': ['ff'],
 'forprene': ['fp'],
 'fortron': ['fo'],
 'frianyl': ['fri'],
 'geolast': ['ge'],
 'hostaform': ['hf'],
 'hytrel': ['hyt', 'htr'],
 'impet': ['im'],
 'kepital': ['kep'],
 'laprene': ['lp'],
 'micromax': ['mcm'],
 'neolast': ['neo'],
 'omnicarb': ['oc'],
 'omnilon': ['ol'],
 'omnipro': ['opp'],
 'omnitech': ['ot'],
 'pibifor': ['po'],
 'pibiter': ['pb'],
 'pipelon': ['pip'],
 'polifor': ['pf'],
 'rynite': ['ryn'],
 'santoprene': ['stp'],
 'sofprene': ['sp'],
 'tarnoform': ['ta'],
 'tecnoprene': ['tc'],
 'thermx': ['tx'],
 'vamac': ['vamc'],
 'vandar': ['va'],
 'vectra': ['ve'],
 'zenite': ['ze'],
 'zytel': ['zyt']}

In [13]:
len(all_brands)

19

In [14]:
for i in list(brand_synonyms.values()):
    all_brands.extend(i)
    
with open("all_brands_" + VERSION + ".json", "w") as fp:
    json.dump(all_brands, fp)

In [15]:
all_brands

['celanex',
 'celanyl',
 'celcon',
 'celstran',
 'coolpoly',
 'crastin',
 'ecomid',
 'fortron',
 'frianyl',
 'gur',
 'hostaform',
 'hytrel',
 'kepital',
 'rynite',
 'santoprene',
 'thermx',
 'vectra',
 'zenite',
 'zytel',
 'am',
 'bf',
 'br',
 'blue ridge',
 'cx',
 'cnl',
 'cl',
 'cn',
 'cs',
 'lft',
 'lfrt',
 'cs lft',
 'celstran lft',
 'pultrusion',
 'lftr',
 'cp',
 'cra',
 'mid',
 'fa',
 'ff',
 'fp',
 'fo',
 'fri',
 'ge',
 'hf',
 'hyt',
 'htr',
 'im',
 'kep',
 'lp',
 'mcm',
 'neo',
 'oc',
 'ol',
 'opp',
 'ot',
 'po',
 'pb',
 'pip',
 'pf',
 'ryn',
 'stp',
 'sp',
 'ta',
 'tc',
 'tx',
 'vamc',
 'va',
 've',
 'ze',
 'zyt']

### Competitor Grades

In [16]:
Competitor_grades = read_data_from_snowflake_table(cur,"""select distinct competitor_grade from GST_Curated.competitor_data""")
Competitor_grades = Competitor_grades['competitor_grade'].unique().tolist()
for i in Competitor_grades:
 if 'inte' in i.lower():
    print(i)

Competitor_grades = [re.sub(r'^\((.*)\)$', r'\1', value) for value in Competitor_grades]
Competitor_grades = [value.lower().replace('®', '').replace('™', '').strip() for value in Competitor_grades]

brackets_words = []
for i in Competitor_grades:
    if len(i.split("(")) > 1:
        if 'inte' in i.split("(")[-1]:
            print(i)
        brackets_words.append(i.split("(")[-1])

brackets_words = list(set(brackets_words))

remove_suffixes = ['(ok1)', '(extra)', '(f1 apply gy/bk only)', '(r&d sample)', '(short version)', '(complete data)', '(before new yc)', '(extrusion)', '(condensed data)', '(r&d trial sample)', '(spcl)', '(eu)', '(us)', '(inte)', '(inte', '(condensed)', '(simplified)', '(dev)', '(old version)', '(old)', '(developmental)', ' - asia', ' - europe', ' - americas']
remove_suffixes += [
    '(color).-3',
    '(ptfe)',
]
temp = []
for cg in Competitor_grades:   
    for suffix in remove_suffixes:
        cg = re.sub(r'{}$'.format(re.escape(suffix.lower())), '', cg.lower())
    temp.append(re.sub("(\s+)", " ", cg.replace('®', '').replace('™', '').strip())) 

Competitor_grades = [value.strip() for value in temp]

print(len(Competitor_grades))

Sinterline® Powder PA6 3400 HT 110 NATURAL
Wellamid® MRGF25/15 42H-BK848 (Inte
wellamid mrgf25/15 42h-bk848 (inte
12929


In [17]:
brackets_words = []
for i in Competitor_grades:
    if len(i.split("(")) > 1:
        if 'inte' in i.split("(")[-1]:
            print(i)
        brackets_words.append(i.split("(")[-1])

brackets_words = list(set(brackets_words))
brackets_words

['21006)',
 '3397)',
 '7811)',
 '8148)',
 'pa/pp) m/mo 7101 gf25',
 '6567)',
 '4817)',
 '6569)',
 '3446)',
 '2469)',
 '016)',
 '1830)',
 '6303)',
 '1383)',
 '4916)',
 '7925)',
 '2788)',
 '6924)',
 '6673)',
 '20016)',
 '8112)',
 '8000)',
 '3051)',
 '6412)',
 '6522)',
 '7622)',
 '6665)',
 '2845)',
 '6772)',
 '2310)',
 '7737)',
 '7778)',
 '7486)',
 '5296)',
 '7722)',
 '7825)',
 '7481)',
 '7898)',
 '3147)',
 '7517)',
 '3671)',
 '7923)',
 '6507)',
 '2395)',
 '20028)',
 '4668)',
 '4002)',
 '7381)',
 'fc-x9201)',
 '7489)',
 '3028)',
 '8004)',
 '8149)',
 '1043)',
 '3715)',
 '1259)',
 '3484)',
 '7746)',
 '5202)',
 '9966)',
 '8162)',
 '4363)',
 '5966)',
 'k)',
 '1270)',
 '3695)',
 '6546)',
 '9984)',
 '1802)',
 '7282)',
 '7458)',
 '9987)',
 '3706)',
 '3396)',
 '2474)',
 '1824)',
 '5930)',
 '7653)',
 '3292)',
 '2196)',
 '1365)',
 '3673)',
 'db)',
 '9975)',
 '8051)',
 '5637)',
 '5928)',
 '2315)',
 'pa/pp) m/mo gf 8',
 '2309)',
 '5738)',
 '4835)',
 '6020)',
 '3669)',
 '6437)',
 '7749)',
 '4701)',
 '

In [18]:
alpha_num = []
num = []
for i in brackets_words:
    try: 
        int(i.strip(')'))
        num.append('(' + i)
    except:
        alpha_num.append('(' + i)
    # if i.isalpha(): 
    #     alpha_words.append(i)
    # elif i.isalnum():
    #     alpha_num_words.append(i)

In [19]:
num

['(21006)',
 '(3397)',
 '(7811)',
 '(8148)',
 '(6567)',
 '(4817)',
 '(6569)',
 '(3446)',
 '(2469)',
 '(016)',
 '(1830)',
 '(6303)',
 '(1383)',
 '(4916)',
 '(7925)',
 '(2788)',
 '(6924)',
 '(6673)',
 '(20016)',
 '(8112)',
 '(8000)',
 '(3051)',
 '(6412)',
 '(6522)',
 '(7622)',
 '(6665)',
 '(2845)',
 '(6772)',
 '(2310)',
 '(7737)',
 '(7778)',
 '(7486)',
 '(5296)',
 '(7722)',
 '(7825)',
 '(7481)',
 '(7898)',
 '(3147)',
 '(7517)',
 '(3671)',
 '(7923)',
 '(6507)',
 '(2395)',
 '(20028)',
 '(4668)',
 '(4002)',
 '(7381)',
 '(7489)',
 '(3028)',
 '(8004)',
 '(8149)',
 '(1043)',
 '(3715)',
 '(1259)',
 '(3484)',
 '(7746)',
 '(5202)',
 '(9966)',
 '(8162)',
 '(4363)',
 '(5966)',
 '(1270)',
 '(3695)',
 '(6546)',
 '(9984)',
 '(1802)',
 '(7282)',
 '(7458)',
 '(9987)',
 '(3706)',
 '(3396)',
 '(2474)',
 '(1824)',
 '(5930)',
 '(7653)',
 '(3292)',
 '(2196)',
 '(1365)',
 '(3673)',
 '(9975)',
 '(8051)',
 '(5637)',
 '(5928)',
 '(2315)',
 '(2309)',
 '(5738)',
 '(4835)',
 '(6020)',
 '(3669)',
 '(6437)',
 '(7749)

In [20]:
alpha_num 

['(pa/pp) m/mo 7101 gf25',
 '(fc-x9201)',
 '(k)',
 '(db)',
 '(pa/pp) m/mo gf 8',
 '(fc-x9224)',
 '(k-x08836)',
 '(w)',
 '(h)',
 '(natural)',
 '(f-x08057)',
 '(h) lv',
 '(xa 1701)',
 '(fc-x9207)',
 '(bk22003)',
 '(pa/pp) m/mo 5101',
 '(fc-x9200)',
 '(n-x174)',
 '(k-x0902)',
 '(black)',
 '(l9)',
 '(h) mv',
 '(pa/pp) m/mo gf 25',
 '(pc/pbt)',
 '(pa/pp) m/mo 7101 gf8',
 '(hv) h black',
 '(x)/47',
 '(hv) h natural',
 '(k-x08203)',
 '(p1120d)',
 '(f-x9307)',
 '(pa/pp) m/mo',
 '(x)',
 '(f1)']

In [21]:
for i in Competitor_grades:
    if '7622' in i:
        print(i)

precite p3 gf 30 4 black (7622)


In [22]:
num_suffix_cgrades = []
alpha_num_suffix_cgrades = []
for i in num:
    for cg in Competitor_grades:
        if i in cg:
            num_suffix_cgrades.append(cg)

for i in alpha_num:
    for cg in Competitor_grades:
        if i in cg:
            alpha_num_suffix_cgrades.append(cg)
    

In [23]:
num_suffix_cgrades

['akromid a3 gf 50 3 black (21006)',
 'akromid b3 gf 45 1 black (3397)',
 'precite p3 black (7811)',
 'akroloy pa gf 40 hu natural (8148)',
 'akromid a3 gf 30 hu black (6567)',
 'akromid a3 gm 20/10 s1 black (4817)',
 'akromid a3 gf 50 1 grey (6569)',
 'akromid b3 gf 45 1 black (3446)',
 'akromid b3 gf 15 natural (2469)',
 'compadur 125gf 30 natural (016)',
 'akromid b3 gm 15/15 black (1830)',
 'akromid a3 gf 50 hu black (6303)',
 'akromid b3 gf 30 s1 natural (1383)',
 'akromid a3 gf 15 1 l natural (4916)',
 'akromid b3 gf 35 6 eco black (7925)',
 'akromid a3 gf 13 s3 natural (2788)',
 'precite p3 gf 30 black (6924)',
 'akromid b3 gf 30 6 black (6673)',
 'akromid b3 gf 40 1 black (20016)',
 'akromid a3 gf 30 8 hu black (8112)',
 'akromid t5 gf 50 6 black (8000)',
 'akromid a3 1 fr black (3051)',
 'akroloy pa gf 30 natural (6412)',
 'akromid a3 gf 40 8 black (6522)',
 'precite p3 gf 30 4 black (7622)',
 'akromid b3 gf 30 frt black (6665)',
 'akroloy pa gf 40 black (2845)',
 'akromid t5 

In [24]:
alpha_num_suffix_cgrades

['schulablend (pa/pp) m/mo 7101 gf25',
 'akulon xs36-e2 (fc-x9201)',
 'kebaform c 3090(k)',
 'lumid lw4409a(k)',
 'kebater pbt a9030 grey 037091 (db)',
 'kebaform c 90.0 green 6018 (db)',
 'schulablend (pa/pp) m/mo gf 8',
 'akulon xs36-c1 (fc-x9224)',
 'akulon fuel lock flx-lp (k-x08836)',
 'lumid gp2150a(w)',
 'lumid gp1000b(w)',
 'lumid hm2604a(w)',
 'lumid gp1200a(w)',
 'lumid gp2330b(w)',
 'lumid gp2130a(w)',
 'lumid gp2330a(w)',
 'lumid eg2309b(w)',
 'lumid lw4403a(w)',
 'lumid gp2130b(w)',
 'lumid gp2339b(w)',
 'lumid hi1202a(w)',
 'lumid gp2200b(w)',
 'lumid hi2202b(w)',
 'lumid hi1202b(w)',
 'lumid hi1102a(w)',
 'lumid gp2500b(w)',
 'lumid gp3200b(w)',
 'lumid hi1102b(w)',
 'lumid sg2300b(w)',
 'lumid gp2500a(w)',
 'lumid hi2302a(w)',
 'lumid gp2430a(w)',
 'lumid gp2200a(w)',
 'lumid gp2430b(w)',
 'lumid gp1300a(w)',
 'lumid gp2309a(w)',
 'lumid gp1100a(w)',
 'lumid hi2152a(w)',
 'lumid gp2300a(w)',
 'technomid 66 1(h) lv',
 'technomid 6 1(h) lv',
 'technomid 6 gf30 1 (h)',
 't

In [25]:
len(set(num_suffix_cgrades)) == len(num_suffix_cgrades)

True

In [26]:
sorted(num_suffix_cgrades)

['akroloy pa gf 30 8 black (4171)',
 'akroloy pa gf 30 8 black (6730)',
 'akroloy pa gf 30 black (2718)',
 'akroloy pa gf 30 black (6415)',
 'akroloy pa gf 30 natural (3177)',
 'akroloy pa gf 30 natural (6412)',
 'akroloy pa gf 40 black (2845)',
 'akroloy pa gf 40 black (6416)',
 'akroloy pa gf 40 hu black (8149)',
 'akroloy pa gf 40 hu natural (8148)',
 'akroloy pa gf 40 natural (3059)',
 'akroloy pa gf 40 natural (6413)',
 'akroloy pa gf 50 8 black (6732)',
 'akroloy pa gf 50 8 natural (6606)',
 'akroloy pa gf 50 black (2706)',
 'akroloy pa gf 50 black (6507)',
 'akroloy pa gf 50 black (6546)',
 'akroloy pa gf 50 grey (3807)',
 'akroloy pa gf 50 hu black (8198)',
 'akroloy pa gf 50 natural (2916)',
 'akroloy pa gf 50 natural (5930)',
 'akroloy pa gf 50 natural (6414)',
 'akroloy pa gf 50 natural (7573)',
 'akroloy pa gf 50 white (5137)',
 'akroloy pa gf 60 7 black (2844)',
 'akroloy pa gf 60 7 natural (2941)',
 'akroloy pa gf 60 8 black (6733)',
 'akroloy pa gf 60 beige (7632)',
 'ak

#### For validation

In [27]:
# brackets_data = {"NUMBERS": sorted(num_suffix_cgrades), "LETTERS/NUMBERS": sorted(alpha_num_suffix_cgrades)}

In [28]:
# with open("Competitor_Grades_with_brackets.json", "w") as fp:
#     json.dump(brackets_data, fp, indent=3)

In [29]:
# region_suffix = [' - asia', ' - europe', ' - americas']
# region_suffix_cgrades = []
# for i in Competitor_grades:
#     for r in region_suffix:
#         if r in i:
#             region_suffix_cgrades.append(i)

# region_suffix_cgrades

In [30]:
# with open("Competitor_Grades_with_Region.json", "w") as fp:
#     json.dump({"Comp_Grades_with_REGION": sorted(region_suffix_cgrades)}, fp, indent=3)

In [31]:
# remove_suffixes = ['(ok1)', '(extra)', '(f1 apply gy/bk only)', '(r&d sample)', '(short version)', '(complete data)', '(before new yc)', '(extrusion)', '(condensed data)', '(r&d trial sample)', '(spcl)', '(eu)', '(us)', '(inte)', '(inte', '(condensed)', '(simplified)', '(dev)', '(old version)', '(old)', '(developmental)', ' - asia', ' - europe', ' - americas']

In [32]:
# with open("remove_suffixes.json", "w") as fp:
#     json.dump({"Suffix": remove_suffixes}, fp, indent=3)

#### Further filtering the Cgrades

In [33]:
for i in ["china", "ford", "impact", "industrial", "bmw", "product", "grade"]:
    for cg in Competitor_grades:
        if i in cg:
            print(cg)

tekumid 6 p ms offgrade


In [34]:
ignore_comp_grades = [
    'tekumid 6 p ms offgrade',
    'akulon fuel lock flx40-hp(k-x0902)',
    'akulon fuel lock flx-lp (k-x08836)',
]
Competitor_grades = [x for x in Competitor_grades if x not in ignore_comp_grades]

In [35]:
for i in ["china", "ford", "impact", "industrial", "bmw", "product", "grade"]:
    for cg in Competitor_grades:
        if i in cg:
            print(cg)

In [36]:
print(len(Competitor_grades))
Competitor_grades = [x for x in Competitor_grades if x not in num_suffix_cgrades]
print(len(Competitor_grades))

12926
12257


In [37]:
for i in Competitor_grades:
    if '  ' in i:
        print(i)

In [38]:
for i in Competitor_grades:
    if '..' in i:
        print(i)

In [39]:
for i in Competitor_grades:
    if '(' in i:
        print(i)

akulon diablo hdt2504bm (k-x08203)
dynaflex g7660-9 (black)
dynaflex g7670-1 (natural)
lumid gp2150a(w)
novamid 1017-ah1 (n-x174)
lumid gp1000b(w)
plastron lft pa6-gf60-01(l9)
lumid hm2604a(w)
dynaflex g7640-9 (black)
dynaflex g7680-1 (natural)
plastron lft pax-gf50-02(l9)
akulon xs32-e2 (fc-x9200)
plastron lft pax-gf60-02(l9)
lumid gp1200a(w)
schulablend (pa/pp) m/mo 7101 gf25
lumid gp2330b(w)
lumid gp2130a(w)
lumid gp2330a(w)
dynaflex g7690-1 (natural)
lumid eg2309b(w)
dynaflex g7630-1 (natural)
lumid lw4403a(w)
tepex dynalite 102-rg600(x)/47
bergamid a700 g25 h (f1)
dynaflex g7680-9 (black)
bergamid a700 g40 h (f1)
lumid gp2130b(w)
lumid gp2339b(w)
dynaflex g7660-1 (natural)
dynaflex g7670-9 (black)
lumid hi1202a(w)
technomid 66 1(h) lv
schulablend (pa/pp) m/mo 5101
arnite tz6 280 (bk22003)
technyl 4earth a4e 218 v50 black (xa 1701)
lumid gp2200b(w)
pentalloy bp (hv) h black
lumid hi2202b(w)
dynaflex g7690-9 (black)
lumid hi1202b(w)
lumid hi1102a(w)
lumid gp2500b(w)
bergamid a700 g1

In [40]:
with open("all_cgrades_" + VERSION + ".json", "w") as fp:
    json.dump(Competitor_grades, fp)

In [41]:
Auto_Approvals = [[f"{item['OEM_Name']} {certification}" for certification in item['CERTIFICATIONS']]  for item in unique_values['results']['Auto_Approval']]
other_certifications = [item['CERTIFICATIONS'] for item in unique_values['results']['Certification']]
all_certifications = [item['OEM_Name'] for item in unique_values['results']['Auto_Approval']] + [item for sublist in Auto_Approvals for item in sublist] + [item for sublist in other_certifications for item in sublist]
all_certifications = [value.lower() for value in all_certifications]


with open("all_certifications_" + VERSION + ".json", "w") as fp:
    json.dump(all_certifications, fp)
    
len(all_certifications)

1053